```
esc50/
├── esc50.csv          -- метаданные по клипам
├── audio/44100/*.wav  -- исходные wav-клипы
└── audio/16000/*.wav  -- версия с пониженной частотой дискретизации
```

In [ ]:
from pathlib import Path
import random

import IPython.display as ipd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import soundfile as sf
from tqdm.auto import tqdm

## Setup

In [ ]:
repo_root = Path('.').resolve().parent

metadata_candidates = [
    Path('/kaggle/input/environmental-sound-classification-50/esc50.csv'),
    repo_root / 'data' / 'raw' / 'ESC-50' / 'esc50.csv',
    repo_root / 'data' / 'raw' / 'ESC50' / 'esc50.csv',
    repo_root / 'esc50.csv',
]

audio_44100_candidates = [
    Path('/kaggle/input/environmental-sound-classification-50/audio/audio/44100/'),
    repo_root / 'data' / 'raw' / 'ESC-50' / 'audio' / 'audio' / '44100',
    repo_root / 'data' / 'raw' / 'ESC50' / 'audio' / 'audio' / '44100',
]

audio_16000_candidates = [
    Path('/kaggle/input/environmental-sound-classification-50/audio/audio/16000/'),
    repo_root / 'data' / 'raw' / 'ESC-50' / 'audio' / 'audio' / '16000',
    repo_root / 'data' / 'raw' / 'ESC50' / 'audio' / 'audio' / '16000',
]

metadata_path = next((path for path in metadata_candidates if path.exists()), None)
audio_44100_dir = next((path for path in audio_44100_candidates if path.exists()), None)
audio_16000_dir = next((path for path in audio_16000_candidates if path.exists()), None)

print(f'metadata_path: {metadata_path}')
print(f'audio_44100_dir: {audio_44100_dir}')
print(f'audio_16000_dir: {audio_16000_dir}')

## Download

In [ ]:
download_url = 'https://www.kaggle.com/datasets/mmoreaux/environmental-sound-classification-50'
print(f'ESC-50 download: {download_url}')

print('Download command:')
print(r'''#!/bin/bash
curl -L -o ~/Downloads/environmental-sound-classification-50.zip \
  https://www.kaggle.com/api/v1/datasets/download/mmoreaux/environmental-sound-classification-50''')

## Load Metadata

In [ ]:
if metadata_path is None or audio_44100_dir is None:
    raise FileNotFoundError('Не удалось найти esc50.csv или директорию audio/44100. Проверьте, что датасет скачан.')

esc = pd.read_csv(metadata_path).copy()
esc['audio_path'] = esc['filename'].apply(lambda name: str(audio_44100_dir / name))

broad_group_map = {
    'dog': 'animals', 'rooster': 'animals', 'pig': 'animals', 'cow': 'animals', 'frog': 'animals',
    'cat': 'animals', 'hen': 'animals', 'insects': 'animals', 'sheep': 'animals', 'crow': 'animals',
    'chirping_birds': 'natural', 'water_drops': 'natural', 'wind': 'natural', 'rain': 'natural', 'thunderstorm': 'natural',
    'sea_waves': 'natural', 'crackling_fire': 'natural', 'crickets': 'animals',
    'crying_baby': 'human', 'sneezing': 'human', 'clapping': 'human', 'breathing': 'human', 'coughing': 'human',
    'laughing': 'human', 'snoring': 'human',
    'footsteps': 'interior', 'brushing_teeth': 'interior', 'drinking_sipping': 'interior', 'door_wood_knock': 'interior',
    'mouse_click': 'interior', 'keyboard_typing': 'interior', 'door_wood_creaks': 'interior', 'can_opening': 'interior',
    'toilet_flush': 'interior', 'pouring_water': 'interior',
    'washing_machine': 'domestic', 'vacuum_cleaner': 'domestic', 'clock_alarm': 'domestic', 'clock_tick': 'domestic', 'glass_breaking': 'domestic',
    'helicopter': 'vehicles', 'engine': 'vehicles', 'train': 'vehicles', 'airplane': 'vehicles',
    'chainsaw': 'urban', 'siren': 'urban', 'car_horn': 'urban', 'church_bells': 'urban', 'fireworks': 'urban', 'hand_saw': 'urban',
}

esc['broad_group'] = esc['category'].map(broad_group_map).fillna('other')

print(f'Total samples: {len(esc)}')
print(f'Classes: {esc["category"].nunique()}')
print(f'Folds: {sorted(esc["fold"].unique().tolist())}')
display(esc.head())

In [ ]:
audio_rows = []
for row in tqdm(esc.itertuples(index=False), total=len(esc), desc='Сбор аудио-метаданных'):
    info = sf.info(row.audio_path)
    audio_rows.append(
        {
            'filename': row.filename,
            'category': row.category,
            'target': row.target,
            'fold': row.fold,
            'esc10': bool(row.esc10),
            'src_file': row.src_file,
            'take': row.take,
            'broad_group': row.broad_group,
            'duration_seconds': info.duration,
            'sample_rate': info.samplerate,
            'channels': info.channels,
            'frames': info.frames,
        }
    )

esc_audio = pd.DataFrame(audio_rows)
display(esc_audio[['duration_seconds', 'sample_rate', 'channels', 'frames']].describe().transpose())

## EDA

In [ ]:
sns.set_theme(style='whitegrid', context='talk')

class_counts = esc_audio['category'].value_counts().rename_axis('category').reset_index(name='num_samples')
fold_counts = esc_audio['fold'].value_counts().sort_index().rename_axis('fold').reset_index(name='num_samples')

fig, axes = plt.subplots(1, 2, figsize=(20, 7))
sns.barplot(data=class_counts, x='num_samples', y='category', palette='viridis', ax=axes[0])
axes[0].set_title('Количество клипов по классам')
axes[0].set_xlabel('Число клипов')
axes[0].set_ylabel('Класс')

sns.barplot(data=fold_counts, x='fold', y='num_samples', palette='Set2', ax=axes[1])
axes[1].set_title('Количество клипов по fold')
axes[1].set_xlabel('Fold')
axes[1].set_ylabel('Число клипов')

plt.tight_layout()
plt.show()

### Баланс датасета

ESC-50 очень хорошо сбалансирован: каждый класс содержит одинаковое число клипов, а все пять folds имеют одинаковый размер. Это делает датасет удобным для честного сравнения моделей, потому что качество меньше зависит от перекоса по частотам классов.

In [ ]:
fold_category = esc_audio.groupby(['fold', 'category']).size().unstack(fill_value=0)
esc10_fold = esc_audio.groupby(['fold', 'esc10']).size().reset_index(name='num_samples')

fig, axes = plt.subplots(1, 2, figsize=(20, 7))
sns.heatmap(fold_category, cmap='mako', linewidths=0.25, ax=axes[0])
axes[0].set_title('Распределение классов по fold')
axes[0].set_xlabel('Класс')
axes[0].set_ylabel('Fold')

sns.barplot(data=esc10_fold, x='fold', y='num_samples', hue='esc10', palette='Set1', ax=axes[1])
axes[1].set_title('Распределение ESC-10 поднабора')
axes[1].set_xlabel('Fold')
axes[1].set_ylabel('Число клипов')
axes[1].legend(title='ESC-10')

plt.tight_layout()
plt.show()

### Структура разбиения

Разбиение по fold выглядит строго контролируемым: каждый fold покрывает все классы без заметных перекосов. Это полезно для кросс-валидации и делает ESC-50 удобным эталонным набором для экспериментов с признаками и архитектурами.

In [ ]:
group_counts = esc_audio.groupby('broad_group').size().reset_index(name='num_samples')
group_fold = esc_audio.groupby(['fold', 'broad_group']).size().reset_index(name='num_samples')

fig, axes = plt.subplots(1, 2, figsize=(20, 7))
sns.barplot(data=group_counts, x='broad_group', y='num_samples', palette='crest', ax=axes[0])
axes[0].set_title('Распределение по укрупнённым группам звуков')
axes[0].set_xlabel('Группа')
axes[0].set_ylabel('Число клипов')
axes[0].tick_params(axis='x', rotation=30)

sns.barplot(data=group_fold, x='fold', y='num_samples', hue='broad_group', palette='tab10', ax=axes[1])
axes[1].set_title('Укрупнённые группы по fold')
axes[1].set_xlabel('Fold')
axes[1].set_ylabel('Число клипов')
axes[1].legend(title='Группа', bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.show()

### Семантическое покрытие

Хотя количество клипов по классам одинаковое, по смысловым группам датасет покрывает не всё равномерно. В ESC-50 смешаны животные, природные, бытовые, человеческие и городские звуки, поэтому он хорошо подходит как общий benchmark, но хуже отражает узкую транспортную акустическую область.

In [ ]:
source_counts = esc_audio['src_file'].value_counts().head(20).reset_index()
source_counts.columns = ['src_file', 'num_samples']

unique_sources = (
    esc_audio.groupby('category')['src_file']
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name='num_unique_sources')
)

fig, axes = plt.subplots(1, 2, figsize=(20, 7))
sns.barplot(data=source_counts, x='num_samples', y='src_file', palette='rocket', ax=axes[0])
axes[0].set_title('Топ-20 исходных источников по числу клипов')
axes[0].set_xlabel('Число клипов')
axes[0].set_ylabel('Исходный файл')

sns.barplot(data=unique_sources, x='num_unique_sources', y='category', palette='mako', ax=axes[1])
axes[1].set_title('Число уникальных источников внутри класса')
axes[1].set_xlabel('Уникальные src_file')
axes[1].set_ylabel('Класс')

plt.tight_layout()
plt.show()

### Разнообразие источников

Баланс по классам не означает одинаковую сложность по источникам. Внутри некоторых классов исходных записей `src_file` больше, чем в других, поэтому реальная вариативность акустических условий может отличаться даже при одинаковом числе итоговых клипов.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 7))

sns.boxplot(data=esc_audio, x='duration_seconds', y='category', palette='viridis', ax=axes[0])
axes[0].set_title('Длительность клипов по классам')
axes[0].set_xlabel('Длительность, секунд')
axes[0].set_ylabel('Класс')

sr_counts = esc_audio['sample_rate'].value_counts().sort_index().reset_index()
sr_counts.columns = ['sample_rate', 'num_samples']
sns.barplot(data=sr_counts, x='sample_rate', y='num_samples', palette='Set2', ax=axes[1])
axes[1].set_title('Распределение частоты дискретизации')
axes[1].set_xlabel('Sample rate')
axes[1].set_ylabel('Число клипов')

plt.tight_layout()
plt.show()

### Техническая однородность

ESC-50 почти полностью стандартизирован по длительности и формату аудио. Это хорошо для воспроизводимых экспериментов, но одновременно делает датасет более чистым, чем многие реальные звуковые потоки, где длина и качество записи заметно меняются.

In [ ]:
example_files = (
    esc_audio.groupby('broad_group')['filename']
    .first()
    .reset_index()
)

fig, axes = plt.subplots(len(example_files), 2, figsize=(18, 4 * len(example_files)))
if len(example_files) == 1:
    axes = np.array([axes])

for row_idx, row in enumerate(example_files.itertuples(index=False)):
    path = audio_44100_dir / row.filename
    signal, sr = librosa.load(path, sr=None)
    librosa.display.waveshow(signal, sr=sr, ax=axes[row_idx, 0])
    axes[row_idx, 0].set_title(f'Волна: {row.broad_group}')
    axes[row_idx, 0].set_xlabel('Время')
    axes[row_idx, 0].set_ylabel('Амплитуда')

    mel = librosa.feature.melspectrogram(y=signal, sr=sr, n_mels=128)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    librosa.display.specshow(mel_db, sr=sr, x_axis='time', y_axis='mel', ax=axes[row_idx, 1])
    axes[row_idx, 1].set_title(f'Mel-спектрограмма: {row.broad_group}')
    axes[row_idx, 1].set_xlabel('Время')
    axes[row_idx, 1].set_ylabel('Mel')

plt.tight_layout()
plt.show()

display(example_files)

### Визуальные различия сигналов

Даже на уровне нескольких примеров видно, что группы звуков сильно отличаются по временной структуре и частотному наполнению. Это подтверждает, что ESC-50 полезен не только как табличный benchmark по меткам, но и как набор для изучения спектральных признаков и устойчивости представлений.

In [ ]:
sample_row = esc_audio.sample(1, random_state=42).iloc[0]
print(sample_row[['filename', 'category', 'broad_group', 'fold']])
ipd.Audio(sample_row['audio_path'])

### Итоги по датасету

ESC-50 — это аккуратный и очень удобный benchmark для задач классификации экологических звуков.

Его сильные стороны — ровный баланс классов, стабильная длина клипов, понятное fold-разбиение и широкий набор повседневных акустических событий.

Его ограничение в том, что он заметно чище и стандартизированнее многих реальных сценариев. Поэтому для задач доменной адаптации и городской акустики его лучше использовать как базовый ориентир, а не как финальное приближение к production-данным.